In [ ]:
%%writefile submission.py
# #####################################################################
# #                   YOUR REWARD DESIGN PLAYGROUND                   #
# #####################################################################
#
#  WHAT IS THIS?
#  -------------
#  An AI learns by getting rewards for good actions and penalties for
#  bad ones. This `CumulativeRewardWrapper` is where we design those
#  rewards. Think of it as giving the AI "hints" or "breadcrumbs" to
#  help it figure out the best strategy.
#
#  YOUR GOAL:
#  -----------
#  Experiment here! Change the numbers and the logic in the functions
#  below to see if you can train a smarter AI. Get creative as possible
#  as you guide your agent to find and extinguish fires. 
#
# #####################################################################

import gymnasium as gym
from gymnasium import spaces
import numpy as np
import time

#sys.path.append('/kaggle/input/task-1-challenge-competition-2026/env/') 
#sys.path.append('/kaggle/input/task-1-challenge-competition-2026/env/env') 
"""
Gymnasium wrappers for the FireFighting environment.

FlattenObservationWrapper
    Converts the Dict observation space into a flat 1D float32 array
    compatible with standard RL libraries (e.g. Stable-Baselines3).

CumulativeRewardWrapper
    Dense reward shaping on top of the sparse base rewards. Designed for
    UAV_WATER_RANGE = 1 where every douse is an instant extinguish.

    Shaping layers (applied every step unless terminated by crash):
      1. Boundary penalty     — discourages hugging walls

    On a successful termination (all fires out) shaping is NOT zeroed,
    so the agent still receives the positive signal from the final step.
    On a crash termination all shaping is zeroed to keep the crash signal clean.
"""

class FlattenObservationWrapper(gym.ObservationWrapper):
    """Flattens the Dict observation into a 1D float32 array."""

    def __init__(self, env):
        super().__init__(env)
        total_size = sum(
            np.prod(space.shape)
            for space in env.observation_space.spaces.values()
        )
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf,
            shape=(int(total_size),),
            dtype=np.float32,
        )

    def observation(self, obs):
        return np.concatenate(
            [np.asarray(v).flatten() for v in obs.values()]
        ).astype(np.float32)


class CumulativeRewardWrapper(gym.Wrapper):
    """
    Dense reward shaping wrapper.
    --> This is the class you should edit! <--
    """
    def __init__(self, env, curriculum_stage: int = 1):
        super().__init__(env)
        
        # --- Boundary shaping ---
        # This penalizes the AI for getting too close to the walls.
        # It's a "negative reward" to discourage bad behavior.
        # It doesn't necessarily mean that the agent will explore the grid, it may just circle the same area. 
        self.boundary_penalty = -1.0
        self.danger_zone      = 1

    def reset(self, **kwargs):
        # The parent environment's reset is all we need for now.
        return self.env.reset(**kwargs)

    def step(self, action):
        obs, base_reward, terminated, truncated, info = self.env.step(action)
        
        # Calculate our custom reward hints
        shaping = self._compute_shaping(obs, terminated)
        
        # Add our hints to the base reward from the game
        return obs, base_reward + shaping, terminated, truncated, info

    def _compute_shaping(self, obs, terminated: bool) -> float:
        """
        Computes the total shaping reward for this step.
        """
        # We check if the AI crashed. If it did, we give no hints,
        # just the big crash penalty from the base environment.
        all_fires_out = (self.unwrapped.get_fires_extinguished() == len(self.unwrapped.fires))
        is_crash = terminated and not all_fires_out
        if is_crash:
            return 0.0

        # Start with a reward of 0 and add our reward functions. 
        shaping = 0.0
        shaping += self._boundary_shaping(obs)
        # HINT: You should try and make a reward based function that encourages the agent to also explore the grid.
        # The boundary shaping will help it stay within bounds, but it may not even want to explore if there is no reward for it (it'll just circle the same position)
        # There are many other reward/penalty functions you can implement, so use your imagination! 
        # What can we use from our observation space to reward or penalize the agent in order to achieve our objective. 
        
        # ===> ADD YOUR NEW REWARD FUNCTIONS HERE! <===
        # For example:
        # shaping += self._my_exploration_reward(obs)
        
        return shaping

    def _boundary_shaping(self, obs) -> float:
        """Penalises each UAV within danger_zone cells of any wall."""
        grid_size = self.unwrapped.config.GRID_SIZE
        penalty   = 0.0
        for pos in obs["uav_positions"]:
            x, y = int(pos[0]), int(pos[1])
            # We use the grid size and our designated danger zone to penalize the agent if it enters the danger_zone within the grid.
            # we can make this danger_zone higher or lower in our __init__ function above.
            if (x < self.danger_zone or x > grid_size - 1 - self.danger_zone or
                y < self.danger_zone or y > grid_size - 1 - self.danger_zone):
                penalty += self.boundary_penalty
        return penalty

    def _exploration_shaping(self, obs) -> float:
        ### ADD YOUR CODE HERE ###
        reward = 0.0 # replace with your own reward system.
        return reward 

In [ ]:
%%writefile recording_service.py
# ###############################################################
# ###############################################################
# ##                                                           ##
# ##                  DO NOT EDIT THIS CELL                    ##
# ##           (CRITICAL INITIALIZATION CODE HERE)             ##
# ##                                                           ##
# ###############################################################
# ###############################################################

import gymnasium as gym
import sys
sys.path.append('/kaggle/working/')
class my_recorder(gym.wrappers.RecordVideo):

    def close_video_recorder(self):
        """Closes the video recorder if currently recording."""
        if self.recording:
            assert self.video_recorder is not None

            # Need to capture the last frame
            self.video_recorder.capture_frame()
            
            self.video_recorder.close()
        self.recording = False
        self.recorded_frames = 1
# ###############################################################
# ###############################################################
# ##                                                           ##
# ##                  DO NOT EDIT THIS CELL                    ##
# ##           (CRITICAL INITIALIZATION CODE HERE)             ##
# ##                                                           ##
# ###############################################################
# ###############################################################       

In [ ]:
%%writefile scoring_service.py
# ###############################################################
# ###############################################################
# ##                                                           ##
# ##                  DO NOT EDIT THIS CELL                    ##
# ##           (CRITICAL INITIALIZATION CODE HERE)             ##
# ##                                                           ##
# ###############################################################
# ###############################################################  

def normalize_score(score):
    """
    Normalizes a score to a scale of 0 to 100.
    """
    # Based on config, crash/smoke_impaired = -15, timestep penalty = -0.02 over 100 episodes, the worse score possible is
    # if you crash on the final timestep
    min_possible_score = (-15*100) + (-0.02*300*100) 
    # Based on config, find_fire = 5, extinguish_fire = 10, douse_fire = 7 over 100 episodes. Best score possible, find, douse
    # and extinguish fire at the first time step over 100 episodes. 
    max_possible_score = (5*100) + (10*100) + (7*100) 
        
    score = max(min_possible_score, min(score, max_possible_score))
        
    normalized = ((score - min_possible_score) / (max_possible_score - min_possible_score)) * 100
    return round(normalized,3)
    
# ###############################################################
# ###############################################################
# ##                                                           ##
# ##                  DO NOT EDIT THIS CELL                    ##
# ##           (CRITICAL INITIALIZATION CODE HERE)             ##
# ##                                                           ##
# ###############################################################
# ###############################################################       

In [ ]:
%%writefile train_stage.py
# ===================================================================
#                      AI TRAINING SCRIPT
# -------------------------------------------------------------------
#  TO TUNE THE AI:
#  Find the "↓↓↓ EXPERIMENT ZONE ↓↓↓" in the `train_stage`
#  function below and change the values.
#
#  BEYOND THIS SCRIPT:
#  This provides you with a starter PPO agent that you can improve.
#  You are free to make your own training script or use other
#  Reinforcement Learning agents.
# ===================================================================

import sys
import os
sys.path.append('/kaggle/input/task-1-challenge-competition-2026/env/')

from env.firefighting_env import FireFightingEnv
from submission import CumulativeRewardWrapper, FlattenObservationWrapper

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.utils import set_random_seed
from recording_service import my_recorder


    
def make_env(rank, curriculum_stage=1,render_mode=None, seed=0, eval=False):
    """
    A utility function to create a Gymnasium environment instance, configure it for a
    specific curriculum stage, and apply necessary wrappers. This factory pattern is
    required by Stable Baselines3 for creating vectorized environments.

    Returns:
        Callable: A function that, when called, returns an initialized and wrapped environment instance.
    """
    def _init():
        """Initializes and wraps the environment."""

        # Create the base environment configured for the specific curriculum stage
        env = FireFightingEnv(render_mode=render_mode, curriculum_stage=curriculum_stage)

        # Apply the reward shaping wrapper to provide dense, guiding rewards
        env = CumulativeRewardWrapper(env, curriculum_stage=curriculum_stage)

        # Apply the observation wrapper to flatten the dict observation space into a 1D vector
        env = FlattenObservationWrapper(env)

        # Seed the environment for reproducibility
        if render_mode != None:
            record = {}
            record['interval'] = 25
            record['prefix'] = "eval"
            env = my_recorder(env, video_folder="videos/", name_prefix=record['prefix'],
                       episode_trigger=lambda x: x % record['interval'] == 0, disable_logger=True)
        env.reset(seed=seed + rank)
        return env

    # Set the random seed for the entire process
    set_random_seed(seed)
    return _init

def train_stage(stage_name, curriculum_stage, total_timesteps, model_to_load, model_to_save):
    """
    Manages the training process for a single stage of the curriculum.

    This function sets up a vectorized environment, loads a previous model if specified
    (allowing for transfer learning between stages), trains the PPO agent for a
    set number of timesteps, and saves the resulting model.

    Args:
        stage_name (str): A descriptive name for the curriculum stage (e.g., "Obstacle Navigation").
        curriculum_stage (int): The numerical identifier for the curriculum stage.
        total_timesteps (int): The number of timesteps to train the model for in this stage.
        model_to_load (str or None): Path to a .zip file of a pre-trained model to load.
                                     If None, a new model is created.
        model_to_save (str): Path to save the newly trained model .zip file.
    """

    print(f"\n{'=' * 20} STARTING CURRICULUM: STAGE {curriculum_stage} ({stage_name}) {'=' * 20}")

    # Create the environment for the current stage
    env = DummyVecEnv([make_env(i, curriculum_stage=curriculum_stage) for i in range(4)])

    # Load a pre-existing model or create a new one
    if model_to_load and os.path.exists(model_to_load):
        print(f"Loading model from: {model_to_load}")
        model = PPO.load(model_to_load, env=env)
    else:
        # ##################################################################
        # #                  ↓↓↓ EXPERIMENT ZONE ↓↓↓                     #
        # #      This is where you can change the AI's "brain" settings!   #
        # #    Think of it like tuning a character in a video game.        #
        # #    Change the numbers below to see how it learns differently.  #
        # ##################################################################
        model = PPO(
            "MlpPolicy",
            env,
            # Make this a 0 if you don't want anything printing about the training.
            verbose=0,

            # === How the AI Gathers Experience ===
            
            # How many steps the AI takes before it pauses to think and learn.
            # Analogy: Like reading a few pages of a book before summarizing them.
            # (Bigger number = more experience before learning)
            n_steps=2048,

            # When the AI learns, it looks at its experiences in small groups ("batches").
            # This is the size of one group.
            # (Smaller batches can lead to more stable, but slower, learning).
            batch_size=64,

            # How many times the AI reviews its recent experiences to learn from them.
            # Analogy: Like re-reading a chapter multiple times to memorize it for a test.
            n_epochs=10,

            # === How the AI Thinks About Rewards ===
            
            # How much the AI cares about future rewards vs. immediate rewards.
            # (Closer to 1.0 = more "patient", willing to wait for a bigger reward later).
            # (Closer to 0.0 = more "impatient", wants rewards right now).
            gamma=0.99,

            # A tricky one! This helps the AI decide if a good outcome was because of
            # one lucky action or a whole series of smart moves. You can usually leave this one alone.
            gae_lambda=0.95,

            # How much the AI is allowed to change its strategy after each learning session.
            # (A small number prevents it from making crazy, drastic changes).
            clip_range=0.2,

            # === How the AI Explores vs. Exploits ===

            # Encourages the AI to be curious and try new, random things.
            # (Higher value = more random exploration, less sticking to what it knows).
            # (Too high and it will act randomly; too low and it might never find better strategies).
            ent_coef=0.01,
            
            # How big of a "step" the AI takes when updating its strategy.
            # Analogy: Are you taking baby steps or giant leaps to get to the right answer?
            # (Too big and it might overshoot the best strategy; too small and learning is very slow).
            learning_rate=0.0003
        )
        # ##################################################################
        # #                  ↑↑↑ END OF EXPERIMENT ZONE ↑↑↑                #
        # ##################################################################

    # Train the model
    model.learn(total_timesteps=total_timesteps, progress_bar=False) # You can change the progress_bar to True if you want to see it progress within the notebook
                                                                     # NOTE: It needs to be FALSE during submission, otherwise it will fail.  
    # Save the resulting model
    os.makedirs(os.path.dirname(model_to_save), exist_ok=True)
    model.save(model_to_save)
    print(f"Stage complete. Model saved to: {model_to_save}")

    # Clean up the environment
    env.close()


In [ ]:
%%writefile evaluation_stage.py
# ###############################################################
# ###############################################################
# ##                                                           ##
# ##                  DO NOT EDIT THIS CELL                    ##
# ##           (CRITICAL INITIALIZATION CODE HERE)             ##
# ##                                                           ##
# ###############################################################
# ###############################################################

import sys
import os
import numpy as np
import pandas as pd
sys.path.append('/kaggle/input/task-1-challenge-competition-2026/env/')

from env.firefighting_env import FireFightingEnv
from submission import CumulativeRewardWrapper, FlattenObservationWrapper
from train_stage import make_env
from scoring_service import normalize_score

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.utils import set_random_seed

def evaluate_model(model_path, render_mode=None, curriculum_stage=4, num_episodes=100):
    """
    Loads a trained model and evaluates its performance on a specified curriculum stage.

    This function runs the model in a non-training environment for a fixed number of
    episodes and reports detailed statistics about its performance, including success rate,
    hit accuracy, and average episode length.

    Args:
        model_path (str): Path to the saved model .zip file to be evaluated.
        render_mode (str, optional): The mode for rendering. Use 'human' to watch the agent play.
        curriculum_stage (int): The environment stage to evaluate the agent on.
        num_episodes (int): The number of episodes to run for the evaluation.
    """

    print(f"\n{'=' * 20} STARTING EVALUATION {'=' * 20}")
    print(f"--- Loading Model from: {model_path} ---")
    print(f"--- Evaluating on Curriculum Stage: {curriculum_stage} for {num_episodes} episodes ---")

    if not os.path.exists(model_path):
        print(f"Error: Model file not found at {model_path}. Cannot run evaluation.")
        return

    # Load the trained model
    model = PPO.load(model_path)

    # Create a single, non-vectorized environment for evaluation
    eval_env = make_env(rank=0, curriculum_stage=curriculum_stage, render_mode=render_mode, seed=np.random.randint(100000))()

    total_steps = 0
    outcomes = {"SUCCESS": 0, "CRASH": 0, "TIMEOUT": 0}
    fires_extinguished_list = []
    successful_hits = 0
    wasted_hits = 0
    final_scores = []
    for episode in range(num_episodes):
        obs, info = eval_env.reset()
        done, truncated = False, False

        # Reset the facing state tracker in the wrapper if it exists
        if hasattr(eval_env, 'was_facing'):
            eval_env.was_facing.fill(False)

        while not done and not truncated:
            # Use the unwrapped environment to get the true state before the step
            prev_obs_dict = eval_env.unwrapped._get_obs()
            action, _ = model.predict(obs, deterministic=True)

            is_dousing = (action == FireFightingEnv.ACTION_DOUSE)

            obs, reward, done, truncated, info = eval_env.step(action)
            total_steps += 1
            if np.any(is_dousing):
                current_obs_dict = eval_env.unwrapped._get_obs()
                waters_dropped = prev_obs_dict["uav_water_drops"] > current_obs_dict["uav_water_drops"]

                if np.any(waters_dropped):
                    hp_changed = np.any(prev_obs_dict["fire_hps"] > current_obs_dict["fire_hps"])
                    if hp_changed:
                        successful_hits += 1
                    else:
                        wasted_hits += 1
                        
        episode_score = info['score']
        final_scores.append(episode_score)
        eval_env.save_metrics(done, truncated)    

        unwrapped_env = eval_env.unwrapped
        fires_extinguished = unwrapped_env.get_fires_extinguished()
        if fires_extinguished == len(unwrapped_env.fires) and len(unwrapped_env.fires) > 0:
            outcome = "SUCCESS"
        else:
            outcome = "CRASH" if done else "TIMEOUT"
        outcomes[outcome] += 1
        fires_extinguished_list.append(fires_extinguished)
        

    eval_env.close()
    total_score = sum(final_scores)
    total_score = normalize_score(total_score)
    print(f"\n--- Evaluation Complete ---")
    print(f"Total Score over {num_episodes} episodes: {total_score:.2f}")
    
    # --- 3. CREATE SUBMISSION FILE ---
    # This will be saved to /kaggle/working/submission.csv
    submission_df = pd.DataFrame({'id': ['final_score'], 'score': [total_score], 'Usage': 'Public'})
    submission_df.to_csv('submission.csv', index=False)
    
    print("\n'submission.csv' has been created in the output directory.")

    # Just some immediate metrics you can see after a succesful evaluation run
    print("\nOutcomes:")
    for outcome, count in outcomes.items():
        print(f"  - {outcome}: {count}/{num_episodes} ({(count / num_episodes) * 100:.1f}%)")

    print(f"\nAverage fires extinguished per Episode: {np.mean(fires_extinguished_list):.2f}")

    print("\nPerformance:")
    print(f"  - Total Successful hits: {successful_hits}")
    print(f"  - Total Wasted water drops: {wasted_hits}")

    hit_accuracy = 0
    total_hits = successful_hits + wasted_hits
    if total_hits > 0:
        hit_accuracy = successful_hits / total_hits
    print(f"  - Hit Accuracy: {hit_accuracy:.2%}")
    print("-------------------------\n")
    
# ###############################################################
# ###############################################################
# ##                                                           ##
# ##                  DO NOT EDIT THIS CELL                    ##
# ##           (CRITICAL INITIALIZATION CODE HERE)             ##
# ##                                                           ##
# ###############################################################
# ###############################################################

In [ ]:
# #####################################################################
# #              🚀 STAGE 1 & 2: AGENT TRAINING ZONE 🚀               #
# #####################################################################
#
#  WHAT HAPPENS HERE?
#  ------------------
#  This cell trains your AI agent in two stages. First, it learns to
#  navigate, and then it learns to find fires.
#
#  YOUR GOAL:
#  -----------
#  Your main tasks are to:
#  1. **TUNE THE TIMESTEPS:** Adjust `stage1_timesteps` and
#     `stage2_timesteps` below to see how longer or shorter training
#     periods affect the AI's learning.
#  2. **IMPROVE REWARDS:** Enhance the `CumulativeRewardWrapper`
#     in `submission.py` to guide the AI more effectively.
#
# #####################################################################

# --- Boilerplate and Imports ---
import warnings
warnings.filterwarnings("ignore")
import sys
sys.path.append('/kaggle/working/')
from train_stage import train_stage


# ---------------------------------------------------------------------
#   STAGE 1: Learning to Navigate
# ---------------------------------------------------------------------
# In this first stage, your agent learns the basics of moving around
# without crashing. The reward shaping for this is already provided.
print("--- Starting Stage 1: Navigation Training ---")
# ===> EDIT THESE TIMESTEPS TO EXPERIMENT WITH TRAINING DURATION! <===
stage1_timesteps = 400_000
stage1_model_path = "models/cumulative/stage1_navigate.zip"
train_stage(
    stage_name="Navigation",
    curriculum_stage=1,
    total_timesteps=stage1_timesteps,
    model_to_load=None,
    model_to_save=stage1_model_path
)


# ---------------------------------------------------------------------
#   STAGE 2: Learning to Find Fires  (YOUR REWARD TASK IS HERE!)
# ---------------------------------------------------------------------
# Now, using its navigation skills, the agent must learn to find fires.
#
# HINT: To pass this stage, you MUST edit the `CumulativeRewardWrapper`
# in `submission.py`. Add new reward shaping methods to teach your
# agent to gravitate towards fires it discovers!
print("\n--- Starting Stage 2: Fire-Finding Training ---")
# ===> EDIT THESE TIMESTEPS TO EXPERIMENT WITH TRAINING DURATION! <===
stage2_timesteps = 1_500_000
stage2_model_path = "models/cumulative/stage2_find.zip"
train_stage(
    stage_name="Find fire",
    curriculum_stage=2,
    total_timesteps=stage2_timesteps,
    model_to_load=stage1_model_path, # Uses the model from Stage 1. 
    model_to_save=stage2_model_path
)



In [ ]:
# ###############################################################
# ###############################################################
# ##                                                           ##
# ##                  DO NOT EDIT THIS CELL                    ##
# ##           (CRITICAL INITIALIZATION CODE HERE)             ##
# ##                                                           ##
# ###############################################################
# ###############################################################

import sys
import warnings
warnings.filterwarnings("ignore")
# Add the working directory, where your 'submission.py' file must be.
sys.path.append('/kaggle/working/')
from evaluation_stage import evaluate_model

final_model_path =  "models/cumulative/stage2_find.zip"
print("\n--- FINAL EVALUATION ON FULLY RANDOM ENVIRONMENT ---")
# For this task we are testing your agents on curriculum #2, which means there is only 1 fire. 
# Your agent needs to find it and maybe even douse it!
evaluate_model(model_path=final_model_path, curriculum_stage=2, num_episodes=100, render_mode="rgb_array")